# 07 --- MCP Primitives

**CCA Pattern**: MCP (Model Context Protocol) defines three primitives:
- **Tools** -- executable functions an agent can invoke (verbs)
- **Resources** -- data schemas and catalogs agents can query (nouns)
- **Prompts** -- templates for common operations (patterns)

The exam tests whether you know the difference.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.tools.definitions import ALL_TOOL_SETS

## Understanding the Three MCP Primitives

MCP is a protocol that standardizes how AI agents interact with external systems. The CCA exam tests your ability to classify capabilities into the correct primitive.

### The Key Distinction

| Primitive | Nature | Analogy | Example |
|-----------|--------|---------|--------|
| **Tool** | Action (verb) | Function call | `verify_claim(claim)` |
| **Resource** | Data (noun) | Database query | Source reliability ratings |
| **Prompt** | Template (pattern) | Reusable format | Research query decomposition |

The most common exam mistake is classifying a **Resource** as a **Tool**. A source reliability database is *data the agent reads* (Resource), not *an action the agent performs* (Tool).

### How Our System Maps to MCP

Our project uses the `anthropic` SDK directly (not the Agent SDK), so MCP primitives are represented as plain Python constructs. Here's how each maps:

## Tools (Verbs) -- Things the Agent Does

In [ ]:
# Our tool definitions are MCP Tools -- executable functions
for agent_type, tools in ALL_TOOL_SETS.items():
    print(f'{agent_type}:')
    for tool in tools:
        print(f'  Tool: {tool["name"]:<25s} (action/verb)')

Each tool above is an **action** the agent can invoke. `search_web` performs a search. `verify_claim` checks a claim. `delegate_task` triggers delegation. These are verbs.

In MCP terms, each tool definition has:
- A `name` (the function to call)
- A `description` (what it does and does NOT do)
- An `input_schema` (the parameters it accepts)

## Resources (Nouns) -- Things the Agent Reads

In our system, these would be exposed as MCP Resources if we used the MCP protocol:

In [ ]:
# Resources in our system (conceptual mapping)
resources = {
    'document_catalog': 'List of available research documents and their metadata',
    'source_reliability_db': 'Database of source URLs and their reliability ratings',
    'database_schemas': 'Schemas for available statistical tables',
}
for name, desc in resources.items():
    print(f'  Resource: {name:<25s} (data/noun)')

These are **data** the agent queries, not actions it performs. In our codebase:

- `source_reliability_db` -> `KnowledgeBase.get_source_reliability()` returns a SourceReliability enum
- `document_catalog` -> `DocumentStore.list_documents()` returns document metadata
- `database_schemas` -> `DatabaseService.get_schema()` returns column definitions

In a full MCP implementation, these would be exposed as `resource://` URIs that agents can read without executing side effects.

## Prompts (Patterns) -- Templates the Agent Uses

In [ ]:
# Prompts in our system (conceptual mapping)
prompts = {
    'research_query_template': 'Template for decomposing a query into subtasks',
    'citation_format_template': 'Template for formatting citations (APA)',
    'error_report_template': 'Template for reporting structured errors',
}
for name, desc in prompts.items():
    print(f'  Prompt: {name:<30s} (template/pattern)')

In our codebase, these map to:

- `research_query_template` -> The system prompts in `subagents.py` (e.g., `WEB_RESEARCHER_PROMPT`)
- `citation_format_template` -> Would be an MCP Prompt if we needed configurable citation styles
- `error_report_template` -> The `ToolErrorResponse` model structure

In MCP, Prompts are reusable templates that can be invoked with parameters. They're not system prompts -- they're parameterized message templates that standardize common operations.

## Classification Exercise

Five items below. For each, decide whether it is a **Tool**, a **Resource**, or a **Prompt** *before* reading the worked answer in the cell that follows. The CCA exam rewards snap classification, so practice making the call first.

### Item 1 -- `verify_claim(claim: str) -> VerificationResult`

Your classification?

<details>
<summary>Reveal the answer</summary>

**Tool.** It performs an action: checking a claim against the knowledge base. It has side effects (it consumes a `claim` input and produces a `VerificationResult`). Anything shaped like a function call with an imperative verb in the name is almost always a Tool.

Trap to avoid: candidates sometimes classify it as a Resource because it "returns information." Tools also return information -- the distinguishing feature is *computation*, not read-only access.

</details>

### Item 2 -- A database of source URLs with reliability ratings

Your classification?

<details>
<summary>Reveal the answer</summary>

**Resource.** It is read-only data the agent queries. In MCP, this would be exposed as a `resource://source-reliability` URI and the agent would *read* from it rather than *call* it.

This is **the** canonical exam trap. Options include `get_source_reliability(url)` (a Tool-like wrapper) as a distractor. The underlying capability is a Resource even if the access pattern looks Tool-shaped. The exam question is usually worded in terms of the data itself ("a database of ratings"), which is the tell.

</details>

### Item 3 -- A template for decomposing a research query into SubTasks

Your classification?

<details>
<summary>Reveal the answer</summary>

**Prompt.** It is a reusable pattern, parameterized by the incoming query, that produces a structured decomposition. Prompts in MCP are not system prompts -- they are *parameterized message templates* agents can invoke to standardize common operations.

If you said "Tool" because it produces an output: Prompts also produce outputs. The distinguishing feature of a Prompt is that it is a *template* with named parameters, not a function with computation.

</details>

### Item 4 -- `fetch_page(url: str) -> PageContent`

Your classification?

<details>
<summary>Reveal the answer</summary>

**Tool.** It performs the action of fetching a page and has observable side effects (network call, possible failure modes). This is the easiest kind of item on the exam -- clear verb, clear parameter, clear return type. If it can time out or fail, it is almost certainly a Tool (Resources are usually read-only and cacheable in a way that fetch calls are not).

</details>

### Item 5 -- A catalog listing available research documents with metadata

Your classification?

<details>
<summary>Reveal the answer</summary>

**Resource.** A catalog is data the agent browses. The items in it may be things the agent can act on, but the catalog *itself* is a Resource.

Watch this distinction on the exam: `list_documents()` is often listed as a Tool, but the underlying "available documents" is the Resource. If the question asks about the *capability*, go Resource. If it asks about the *function exposed to the agent*, the answer might be Tool. Read carefully.

</details>

## CCA Exam Tip

> The MCP primitives question is a vocabulary test:
> - **Tools** are things the agent *does* (verbs)
> - **Resources** are things the agent *reads* (nouns)
> - **Prompts** are templates the agent *uses* (patterns)
>
> If a question describes a source reliability database -> **Resource**, not a Tool.
> If it describes a function that verifies a claim -> **Tool**.
> If it describes a reusable template for formatting citations -> **Prompt**.
>
> The most common distractor makes a Resource look like a Tool because both involve 'getting information.' The difference: Tools have **side effects** or perform **computation**; Resources are **read-only data**.

*Foolproof test:* **if you cannot *run* it, it is probably a Resource.** A Tool *performs an action* (e.g., `fetch_page(url)`, `verify_claim(claim)`) -- often *using* a Resource. A Resource is the *data or infrastructure the Tool acts upon* (e.g., a database of sources, a schema catalog, a file system). Same information can appear behind either primitive -- ask which side of the verb it sits on.